# m100-setup check

Run this notebook in Colab with the **T4 GPU** runtime (Runtime > Change runtime type > T4 GPU), from the **repo root**.
It does three things:

1. runs `nvidia-smi` and writes the output to `setup/setup.log`
2. downloads the two Intel Lab files into `data/` (git-ignored, never committed)
3. checks the generated-layout scenario and writes `setup/download_check.csv`, one row per dataset

When it finishes, save the notebook back to the repo (File > Save a copy in GitHub) and commit `setup/setup.log` and `setup/download_check.csv`.

In [ ]:
# Get into the repo root. In Colab, clone first if the repo is not already here.
import os, subprocess
REPO_URL = "https://github.com/hanvithalla/intel-lab-wsn-benchmarking.git"
REPO_DIR = "intel-lab-wsn-benchmarking"

if not os.path.isdir("setup"):
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
print("working in", os.getcwd())
os.makedirs("setup", exist_ok=True)
os.makedirs("data", exist_ok=True)

## 1. GPU check

In [ ]:
import subprocess, datetime

res = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
with open("setup/setup.log", "w") as f:
    f.write(f"nvidia-smi run at {datetime.datetime.now(datetime.timezone.utc).isoformat()}\n")
    f.write(res.stdout)
    if res.returncode != 0:
        f.write("nvidia-smi failed: " + res.stderr + "\n")
print(res.stdout)
if res.returncode != 0:
    print("nvidia-smi failed. Switch the runtime to T4 GPU and rerun.")
elif "T4" not in res.stdout:
    print("WARNING: the GPU is not a Tesla T4. Switch the runtime to T4 GPU and rerun before committing setup.log.")

## 2. Intel Lab files

Both files are plain text. The bytes recorded are the size of the files as saved in `data/`.

In [ ]:
import urllib.request

BASE = "https://db.csail.mit.edu/labdata/"
INTEL_FILES = ["connectivity.txt", "mote_locs.txt"]

intel_bytes = 0
intel_ok = True
for name in INTEL_FILES:
    path = os.path.join("data", name)
    try:
        urllib.request.urlretrieve(BASE + name, path)
        size = os.path.getsize(path)
        print(f"{name}: {size} bytes")
        intel_bytes += size
        if size == 0:
            intel_ok = False
    except Exception as e:
        print(f"{name}: download failed ({e})")
        intel_ok = False

# Content checks: 54 motes in mote_locs, about 2,970 parseable rows in connectivity
if intel_ok:
    locs = [l.split() for l in open("data/mote_locs.txt") if l.strip()]
    assert len(locs) == 54, f"expected 54 motes, got {len(locs)}"
    rows = 0
    for l in open("data/connectivity.txt"):
        p = l.split()
        if len(p) == 3:
            int(p[0]); int(p[1]); float(p[2]); rows += 1
    print("motes:", len(locs), "| parseable connectivity rows:", rows)
    assert rows > 2900, "connectivity.txt looks truncated"
print("intel_ok =", intel_ok, "| bytes =", intel_bytes)

## 3. Generated random layout

Nothing is downloaded for this dataset. A row is `ok` only if the generator really ran and the layout it wrote passes these checks: 52 sensors inside the 0.5-40.5 m by 1-31 m box, sink at (19.5, 19).
This is a minimal stand-in used only for this setup check. The real generator comes in m500.

In [ ]:
import random, json

def random52(seed, n=52, xr=(0.5, 40.5), yr=(1.0, 31.0), sink=(19.5, 19.0)):
    rng = random.Random(seed)
    nodes = [(round(rng.uniform(*xr), 2), round(rng.uniform(*yr), 2)) for _ in range(n)]
    return {"deployment": "random52", "seed": seed, "n_sensors": n,
            "sink": list(sink), "nodes": nodes}

os.makedirs("data/layouts", exist_ok=True)
layout = random52(seed=1)
lpath = "data/layouts/random52_s1.json"
with open(lpath, "w") as f:
    json.dump(layout, f)

synth_ok = (
    len(layout["nodes"]) == 52
    and all(0.5 <= x <= 40.5 and 1.0 <= y <= 31.0 for x, y in layout["nodes"])
    and layout["sink"] == [19.5, 19.0]
)
synth_bytes = os.path.getsize(lpath)
print("synth_ok =", synth_ok, "| bytes =", synth_bytes)

## 4. Write `setup/download_check.csv`

In [ ]:
import csv

NAME_INTEL = "intel-berkeley-research-lab-sensor-data-intel-lab-data-connectivity-txt-and-mote-locs-txt"
NAME_SYNTH = "synthetic-random-deployment-scenario-under-the-heinzelman-first-order-radio-model"

rows = [
    (NAME_INTEL, "ok" if intel_ok else "failed", intel_bytes),
    (NAME_SYNTH, "ok" if synth_ok else "failed", synth_bytes),
]
with open("setup/download_check.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["dataset", "status", "bytes"])
    w.writerows(rows)

print(open("setup/download_check.csv").read())
assert all(r[1] == "ok" for r in rows), "a dataset row is not ok; fix it before committing"